# PV Images — Data Preparation
**Unit 3 · Lesson 2 | Fault-type classification**

Read labels → build image paths → stratified split → export.

**Input:** EL image. **Target:** mechanical, thermal, dirt or electric damage.
BW/HDR paths are retained for later experiments. No image preprocessing or training here.

Source: [PV_EL_HDR_BW_Test_DB, version 1](https://doi.org/10.17632/pxyzzkbs4b.1),
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Keep the original CITATION, README and LICENSE with the dataset.

## 1. Setup
Use the already extracted dataset. Set `SOURCE_DIR` to the **inner folder containing
`data` and `images`**. Later lessons reuse the same source and prepared files.

In Colab, `/content` is temporary. Mount Google Drive and use a Drive path if files
must survive runtime resets.

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen
from shutil import copyfileobj
from zipfile import ZipFile
import gdown

import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display


SEED = 42

In [2]:
# url = "https://data.mendeley.com/public-api/zip/pxyzzkbs4b/download/1"
# zip_path = Path("pv_images.zip")
# extract_dir = Path("pv_images")

# request = Request(url, headers={
#     "User-Agent": "Mozilla/5.0",
#     "Referer": "https://data.mendeley.com/datasets/pxyzzkbs4b/1",
# })

# with urlopen(request, timeout=120) as response:
#     with zip_path.open("wb") as output:
#         copyfileobj(response, output)

# with ZipFile(zip_path) as archive:
#     archive.extractall(extract_dir)

# print("Extracted to:", extract_dir.resolve())



In [3]:
file_id = "YOUR_GOOGLE_DRIVE_FILE_ID"
zip_path = "pv_images.zip"

gdown.download(id=file_id, output=zip_path, quiet=False)

with ZipFile(zip_path) as archive:
    archive.extractall("pv_images")

Downloading...
From (original): https://drive.google.com/uc?id=1Wo4RzzhOm8i9s9m09xVgTxnDenYLnhis
From (redirected): https://drive.google.com/uc?id=1Wo4RzzhOm8i9s9m09xVgTxnDenYLnhis&confirm=t&uuid=0ca9ccf1-3e24-4091-b55f-9da835a350b3
To: /content/pv_images.zip
100%|██████████| 1.37G/1.37G [00:19<00:00, 70.1MB/s]


In [4]:
# Optional in Colab:
# from google.colab import drive
# drive.mount("/content/drive")

In [19]:
# EDIT to match your extracted dataset:
SOURCE_DIR = Path("/content/pv_images/PV_EL_HDR_BW_Test_DB")
OUTPUT_DIR = SOURCE_DIR.parent / "prepared_fault_classification"

## 2. Read the label table
Use SMALL only: FULL contains the same observations. The separator is `;`.
Identifiers are read as strings. Match the filename capitalization to your extracted file.

In [20]:
table_path = SOURCE_DIR / "data" / "ELPV_DATA_SMALL.CSV"
raw_df = pd.read_csv(table_path, sep=";", dtype="string")

print("Rows:", len(raw_df))
display(raw_df[["EL_file_name", "incremental_damage_kind"]].head())

Rows: 84


,EL_file_name,incremental_damage_kind
0,0.1_D01_EL_rt_cr.png,unknown
1,0.1_D02_EL_rt_cr.png,mechanical
2,0.1_D03_EL_rt_cr.png,mechanical
3,0.1_D04_EL_rt_cr.png,mechanical
4,0.1_D05_EL_rt_cr.png,mechanical


### Label meaning
`incremental_damage_kind` gives the type of damage added at that stage.
Earlier damage may still be present; this is not a complete inventory of visible defects.

The filename links an image to its label. In `0.1_D02_EL_rt_cr.png`, `0.1` is the
cell, `D02` is a damage-stage ID and `EL` is the modality—not the fault class.
The documentation uses `S#` for soiling and `D##` for other damage stages.
`Degradation_level` is severity, not our target.

The two `unknown` rows have no specified fault type. Keep them in a separate file,
not in the four-class training/evaluation sets. Do not relabel them as healthy.

## 3. Select labeled samples
Keep four specified fault types for classification and save `unknown` separately.
Each row represents one state, not one independent physical cell.

In [21]:
samples = raw_df[
    ["images_set_name", "EL_file_name", "PV_cell_name", "incremental_damage_kind"]
].rename(columns={
    "images_set_name": "sample_id",
    "EL_file_name": "el_filename",
    "PV_cell_name": "cell_id",
    "incremental_damage_kind": "fault_type",
}).copy()

unknown_df = samples.loc[samples["fault_type"] == "unknown"].copy()
labeled_df = samples.loc[samples["fault_type"] != "unknown"].copy()

CLASS_TO_ID = {"mechanical": 0, "thermal": 1, "dirt": 2, "electric": 3}
labeled_df["label"] = labeled_df["fault_type"].map(CLASS_TO_ID).astype("int64")

print("Labeled:", len(labeled_df), "| Unspecified:", len(unknown_df))

Labeled: 82 | Unspecified: 2


## 4. Build image paths
The dataset has 84 states × 3 modalities = 252 images. After excluding two unknown
states, 82 labeled states remain (246 images). Each row retains all three paths;
paired modalities always stay in the same split. EL can be used as a single-modality
baseline; these tables also support later BW/HDR experiments.

Paths are relative to `SOURCE_DIR`. The BW filename token is `BN`.
This step constructs paths; it does not open or verify image files.

In [27]:
samples["el_path"] = "images/EL/" + samples["el_filename"]
samples["bw_path"] = "images/BW/" + samples["el_filename"].str.replace(
    "_EL_", "_BN_"
)
samples["hdr_path"] = "images/HDR/" + samples["el_filename"].str.replace(
    "_EL_", "_HDR_"
)
samples = samples.drop(columns="el_filename")

path_columns = ["sample_id", "el_path", "bw_path", "hdr_path"]
labeled_df = labeled_df.drop(columns="el_filename").merge(samples[path_columns], on="sample_id")
unknown_df = unknown_df.drop(columns="el_filename").merge(samples[path_columns], on="sample_id")
display(labeled_df.head())

KeyError: 'el_filename'

## 5. Train / validation / test: approximately 70 / 15 / 15
Shuffle with a fixed seed and stratify by fault type in both splitting steps.
Integer sample counts mean percentages are approximate.

**Scope:** this is an educational random-split baseline on images from two physical
cells. Different stages of the same cell can appear in different subsets; shared
appearance and residual damage may make scores optimistic. This does not establish
performance on new cells or future stages. No timestamp-based split is assumed.

`cell_id` is retained as metadata, never as a model input. Fit learned preprocessing
on train only, use validation for model selection, and reserve test for final evaluation.
Keep future patches and augmentations in their source image's subset.

In [28]:
train_df, remaining_df = train_test_split(
    labeled_df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=labeled_df["fault_type"],
)
val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=remaining_df["fault_type"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 6. Check subset sizes and class coverage

In [29]:
counts = pd.DataFrame({
    "train": train_df["fault_type"].value_counts(),
    "validation": val_df["fault_type"].value_counts(),
    "test": test_df["fault_type"].value_counts(),
}).reindex(CLASS_TO_ID).fillna(0).astype(int)
display(counts)
print("Subset sizes:", len(train_df), len(val_df), len(test_df))

,train,validation,test
fault_type,,,
mechanical,33,7,7
thermal,11,2,3
dirt,9,2,2
electric,4,1,1


Subset sizes: 57 12 13


## 7. Export
Save the split once and reuse these tables in later lessons—do not independently
split again. Source files are unchanged. `unknown` samples remain outside evaluation.

In [30]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(OUTPUT_DIR / "pv_images_train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "pv_images_validation.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "pv_images_test.csv", index=False)
unknown_df.to_csv(OUTPUT_DIR / "pv_images_unknown.csv", index=False)

print("Saved to:", OUTPUT_DIR)

Saved to: /content/pv_images/prepared_fault_classification
